In [0]:
from pyspark.sql.functions import col

actual_df = spark.table("crypto_ml.cryptosilver.btc_features") \
    .select(
        col("date"),
        col("close").cast("double").alias("close_price"),
        col("volume").cast("double").alias("volume"),
        col("rsi14").cast("double").alias("rsi14"),
        col("return_1d").cast("double").alias("return_1d")
    ) \
    .orderBy(col("date").asc())

display(actual_df)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col

lr_dash = spark.table("crypto_ml.cryptogold.btc_lr_predictions") \
    .select(
        col("date"),
        col("actual_close").cast("double").alias("actual_close"),
        col("predicted_close").cast("double").alias("predicted_close")
    ) \
    .orderBy(col("date").asc())

display(lr_dash)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col

arima_dash = spark.table("crypto_ml.cryptogold.btc_arima_predictions") \
    .select(
        col("date"),
        col("predicted_close").cast("double").alias("predicted_close")
    ) \
    .orderBy(col("date").asc())

display(arima_dash)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col

forecast_table = spark.table("crypto_ml.cryptogold.btc_arima_predictions") \
    .orderBy(col("date").desc()) \
    .limit(7)

display(forecast_table)

In [0]:
from pyspark.sql.functions import col, lit

actual_for_union = spark.table("crypto_ml.cryptosilver.btc_features") \
    .select(
        col("date"),
        col("close").cast("double").alias("price")
    ) \
    .withColumn("series", lit("Actual"))

arima_for_union = spark.table("crypto_ml.cryptogold.btc_arima_predictions") \
    .select(
        col("date"),
        col("predicted_close").cast("double").alias("price")
    ) \
    .withColumn("series", lit("Forecast"))

combined_trend = actual_for_union.unionByName(arima_for_union) \
    .orderBy(col("date").asc())

display(combined_trend)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import max as spark_max

latest_actual_date = spark.table("crypto_ml.cryptosilver.btc_features") \
    .agg(spark_max("date")).collect()[0][0]

latest_actual = spark.sql(f"""
    SELECT date, close
    FROM crypto_ml.cryptosilver.btc_features
    WHERE date = '{latest_actual_date}'
""")

display(latest_actual)

In [0]:
from pyspark.sql.functions import max as spark_max

latest_forecast_date = spark.table("crypto_ml.cryptogold.btc_arima_predictions") \
    .agg(spark_max("date")).collect()[0][0]

latest_forecast = spark.sql(f"""
    SELECT date, predicted_close
    FROM crypto_ml.cryptogold.btc_arima_predictions
    WHERE date = '{latest_forecast_date}'
""")

display(latest_forecast)